# M3 Profile: Lichess arena tournaments
Reads the M2 database **read-only** (`mode=ro`), so nothing here can edit raw data. Run top to bottom on a fresh `python src/load.py` build.
Findings are numbered **I-xx** and feed `docs/cleaning_report.md` (issues register). Families: 1 missing, 2 invalid, 3 unusual, 4 sequence/cross-field, 5 text, 6 identity.

In [1]:
import sqlite3, unicodedata, os, json
import pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 100); pd.set_option("display.max_colwidth", 70)

CANDIDATES = ["data/lichess.db", "lichess.db", "../data/lichess.db", "/mnt/user-data/uploads/lichess.db"]   # first one that exists
DB_PATH = next(p for p in CANDIDATES if os.path.exists(p))
con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

def q(sql, params=()):
    return pd.read_sql(sql, con, params=params)

tables = q("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name")["name"].tolist()
print("database:", DB_PATH)
pd.Series({t: con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0] for t in tables}, name="rows").to_frame()

database: ../data/lichess.db


,rows
game,4535
opening,3332
opening_line,3865
player,1307
quarantine_game,4
tournament,278


In [2]:
t = q("SELECT * FROM tournament"); g = q("SELECT * FROM game"); p = q("SELECT * FROM player")
o = q("SELECT * FROM opening"); ol = q("SELECT * FROM opening_line"); qg = q("SELECT * FROM quarantine_game")
print("strict tables:", q("SELECT name, strict FROM pragma_table_list WHERE schema='main' AND name NOT LIKE 'sqlite_%'").set_index("name").strict.to_dict())

strict tables: {'quarantine_game': 1, 'game': 1, 'opening_line': 1, 'tournament': 1, 'player': 1, 'opening': 1}


## 1. Missing values: count, share, empty strings (every table)

In [3]:
def missing_profile(df):
    rows = []
    for c in df.columns:
        s = df[c]
        empty = int((s.astype("string").str.strip() == "").sum()) if s.dtype == object else 0
        rows.append({"column": c, "dtype": str(s.dtype), "null": int(s.isna().sum()), "null_share": round(s.isna().mean(), 4),
                     "empty_str": empty, "distinct": s.nunique(dropna=True)})
    return pd.DataFrame(rows)
for name, df in [("tournament", t), ("game", g), ("player", p), ("opening", o), ("opening_line", ol), ("quarantine_game", qg)]:
    print("==", name, len(df), "rows"); display(missing_profile(df))

== tournament 278 rows


,column,dtype,null,null_share,empty_str,distinct
0,tournament_key,int64,0,0.0000,0,278
1,source_id,str,0,0.0000,0,278
2,full_name,str,0,0.0000,0,50
3,starts_at,str,0,0.0000,0,278
4,finishes_at,str,0,0.0000,0,247
5,minutes,int64,0,0.0000,0,10
6,clock_limit,int64,0,0.0000,0,8
7,clock_increment,int64,0,0.0000,0,5
8,variant,str,0,0.0000,0,9
9,max_rating,float64,199,0.7158,0,4


== game 4535 rows


,column,dtype,null,null_share,empty_str,distinct
0,game_id,str,0,0.0000,0,4535
1,tournament_key,int64,0,0.0000,0,7
2,white_player_key,int64,0,0.0000,0,1193
3,black_player_key,int64,0,0.0000,0,1195
4,opening_key,float64,387,0.0853,0,611
5,played_at,str,0,0.0000,0,465
6,result,str,0,0.0000,0,3
7,white_elo,int64,0,0.0000,0,1258
8,black_elo,int64,0,0.0000,0,1261
9,white_rating_diff,int64,0,0.0000,0,103


== player 1307 rows


,column,dtype,null,null_share,empty_str,distinct
0,player_key,int64,0,0.0,0,1307
1,source_id,str,0,0.0,0,1307
2,retrieved_at,str,0,0.0,0,3


... (2 outputs left out)

== opening_line 3865 rows


,column,dtype,null,null_share,empty_str,distinct
0,opening_key,int64,0,0.0,0,3332
1,pgn,str,0,0.0,0,3865
2,retrieved_at,str,0,0.0,0,3


== quarantine_game 4 rows


,column,dtype,null,null_share,empty_str,distinct
0,quarantine_id,int64,0,0.0,0,4
1,game_id,str,0,0.0,0,4
2,reason,str,0,0.0,0,1
3,raw_record,str,0,0.0,0,4
4,retrieved_at,str,0,0.0,0,2


### 1a. Coverage: which tournaments have games? (rows per entity)
Only a handful of tournaments have games. Any per-tournament comparison must account for that.

In [4]:
n_games = g.groupby("tournament_key").size()
t["n_games"] = t.tournament_key.map(n_games).fillna(0).astype(int)
print("tournaments:", len(t), " with games:", (t.n_games > 0).sum(), " without:", (t.n_games == 0).sum())
print(pd.crosstab(t.status, t.n_games > 0, rownames=["status"], colnames=["has games"]))
display(t[t.n_games > 0][["tournament_key", "source_id", "full_name", "variant", "status", "nb_players", "n_games", "starts_at"]])
print("games per tournament:", n_games.describe().round(1).to_dict())

tournaments: 278  with games: 7  without: 271
has games  False  True 
status                 
10           209      0
20            29      0
30            33      7


,tournament_key,source_id,full_name,variant,status,nb_players,n_games,starts_at
116,117,mtiiXS2h,Hourly UltraBullet Arena,Standard,30,113,608,2026-10-08T11:30:40Z
117,118,v1YmK7Zs,Hourly Chess960 Arena,Chess960,30,40,101,2026-10-08T11:30:30Z
118,119,ninozBxA,Hourly Bullet Arena,Standard,30,364,950,2026-10-08T11:30:20Z
119,120,RLpiBkWq,Hourly Atomic Arena,Atomic,30,64,237,2026-10-08T11:30:10Z
120,121,5jxGFJv7,≤2000 Bullet Arena,Standard,30,82,184,2026-10-08T11:30:00Z
259,260,NXbdCUqM,Hourly Bullet Arena,Standard,30,463,1333,2026-10-09T17:30:20Z
272,273,vFMisKxe,Hourly Bullet Arena,Standard,30,423,1122,2026-10-09T17:00:05Z


games per tournament: {'count': 7.0, 'mean': 647.9, 'std': 495.0, 'min': 101.0, '25%': 210.5, '50%': 608.0, '75%': 1036.0, 'max': 1333.0}


### 1b. nb_players = 0 and status: placeholder vs measurement (I-01, I-02)
`status` 10 = created, 20 = started, 30 = finished. A table snapshot taken before/during a tournament cannot hold the final player count.

In [5]:
last_retr = t.retrieved_at.max()
print("latest tournament retrieved_at:", last_retr)
print(pd.crosstab(t.status, t.retrieved_at, rownames=["status"], colnames=["retrieved_at"]))
print("nb_players == 0 by status:", t[t.nb_players == 0].status.value_counts().to_dict())
print("status 10 with nb_players > 0 (pre-registration):", int(((t.status == 10) & (t.nb_players > 0)).sum()))
stale = (t.status != 30) & (t.finishes_at < last_retr)
print("status != 30 total:", int((t.status != 30).sum()), "| of those, tournament already over at latest retrieval (stale snapshot):", int(stale.sum()))
print(t[stale].groupby(["status", "retrieved_at"]).size())
print("status 30 rows with finishes_at after their retrieved_at (should be 0):", int(((t.status == 30) & (t.finishes_at > t.retrieved_at)).sum()))

latest tournament retrieved_at: 2026-10-09T18:19:39Z
retrieved_at  2026-10-08T12:06:52Z  2026-10-09T18:19:39Z
status                                                  
10                             102                   107
20                              14                    15
30                              20                    20
nb_players == 0 by status: {10: 182}
status 10 with nb_players > 0 (pre-registration): 27
status != 30 total: 238 | of those, tournament already over at latest retrieval (stale snapshot): 116
status  retrieved_at        
10      2026-10-08T12:06:52Z    102
20      2026-10-08T12:06:52Z     14
dtype: int64
status 30 rows with finishes_at after their retrieved_at (should be 0): 0


### 1c. max_rating is NULL for 199 tournaments (I-03): not applicable, or missing?

In [6]:
t["capped_name"] = t.full_name.str.contains("≤")
print(pd.crosstab(t.max_rating.isna(), t.capped_name, rownames=["max_rating NULL"], colnames=["name has ≤"]))
mr = t[t.max_rating.notna()]
print("cap in name equals max_rating for all capped tournaments:", bool((mr.full_name.str.extract(r"≤(\d+)")[0].astype(int) == mr.max_rating).all()))

name has ≤       False  True 
max_rating NULL              
False                0     79
True               199      0
cap in name equals max_rating for all capped tournaments: True


### 1d. game.opening_key is NULL for 387 games (I-04)

In [7]:
gv = g.merge(t[["tournament_key", "variant"]], on="tournament_key")
display(pd.crosstab(gv.variant, gv.opening_key.isna(), rownames=["variant"], colnames=["opening NULL"]))
print("quarantined games (M2 loader dropped them):", len(qg), qg.reason.value_counts().to_dict())
display(pd.DataFrame([{"game_id": r.game_id, "ECO": json.loads(r.raw_record).get("ECO"), "Opening": json.loads(r.raw_record).get("Opening")} for r in qg.itertuples()]))

opening NULL,False,True
variant,,
Atomic,0,237
Chess960,0,101
Standard,4148,49


quarantined games (M2 loader dropped them): 4 {'unknown_opening': 4}


,game_id,ECO,Opening
0,tSVZ2t6d,C11,"French Defense: Classical Variation, Burn Variation"
1,7FjJQl6X,A52,"Indian Defense: Budapest Defense, Adler Variation"
2,Uuy2CtOo,A52,"Indian Defense: Budapest Defense, Rubinstein Variation"
3,Rh58c9jP,A51,Indian Defense: Budapest Defense


## 2 & 3. Numeric columns: min / median / max, sentinels, negatives, robust outlier screens

In [8]:
SENTINELS = [-999, -99, -1, 0, 9999, 99999]
def numeric_profile(df):
    out = []
    for c in df.select_dtypes("number").columns:
        if c.endswith("_key") or c in ("retrieved_at",): continue
        s = df[c].dropna()
        if s.empty: continue
        med = s.median(); mad = (s - med).abs().median(); q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
        out.append({"column": c, "min": s.min(), "median": med, "max": s.max(), "mad": mad,
                    "negatives": int((s < 0).sum()), "zeros": int((s == 0).sum()),
                    "sentinel_hits": {k: int((s == k).sum()) for k in SENTINELS if (s == k).any()},
                    "top_values": s.value_counts().head(3).to_dict(),
                    "iqr_flagged": int(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()),
                    "robust_z_flagged": int(((0.6745*(s-med)/mad).abs() > 3.5).sum()) if mad > 0 else None})
    return pd.DataFrame(out)
for name, df in [("tournament", t), ("game", g)]:
    print("==", name); display(numeric_profile(df))

== tournament


,column,min,median,max,mad,negatives,zeros,sentinel_hits,top_values,iqr_flagged,robust_z_flagged
0,minutes,27.0,57.0,600.0,30.0,0,0,{},"{27: 133, 57: 119, 60: 10}",11,2.0
1,clock_limit,0.0,60.0,1200.0,45.0,0,5,{0: 5},"{60: 105, 180: 75, 300: 32}",24,56.0
2,clock_increment,0.0,0.0,10.0,0.0,0,220,{0: 220},"{0: 220, 2: 38, 1: 15}",58,NaN
3,max_rating,1300.0,1500.0,2000.0,200.0,0,0,{},"{1300.0: 20, 1500.0: 20, 2000.0: 20}",0,0.0
4,nb_players,0.0,0.0,1893.0,0.0,0,182,{0: 182},"{0: 182, 1: 17, 2: 5}",64,NaN
5,status,10.0,10.0,30.0,0.0,0,0,{},"{10: 209, 30: 40, 20: 29}",69,NaN
6,n_games,0.0,0.0,1333.0,0.0,0,271,{0: 271},"{0: 271, 608: 1, 101: 1}",7,NaN


== game


,column,min,median,max,mad,negatives,zeros,sentinel_hits,top_values,iqr_flagged,robust_z_flagged
0,white_elo,440,1845.0,2721,208.0,0,0,{},"{2010: 14, 1772: 12, 1770: 12}",46,10.0
1,black_elo,440,1843.0,2721,209.0,0,0,{},"{1985: 15, 1967: 14, 1727: 13}",48,10.0
2,white_rating_diff,-190,0.0,169,4.0,2019,375,"{-1: 276, 0: 375}","{0: 375, 1: 336, 2: 284}",111,90.0
3,black_rating_diff,-424,0.0,393,4.0,2152,365,"{-1: 339, 0: 365}","{0: 365, -1: 339, 1: 304}",97,74.0
4,white_berserk,0,0.0,1,0.0,0,4239,{0: 4239},"{0: 4239, 1: 296}",296,NaN
5,black_berserk,0,0.0,1,0.0,0,4253,{0: 4253},"{0: 4253, 1: 282}",282,NaN


### 3a. Unusual ratings and rating changes (I-05, I-06): per variant, robust z (median/MAD)
Rating pools differ by variant, so level is computed per variant. Flagged values are inspected, not deleted.

In [9]:
long = pd.concat([
    gv[["game_id", "variant", "white_elo", "white_rating_diff", "white_player_key", "played_at", "tournament_key"]].set_axis(["game_id", "variant", "elo", "diff", "pk", "at", "tk"], axis=1).assign(side="white"),
    gv[["game_id", "variant", "black_elo", "black_rating_diff", "black_player_key", "played_at", "tournament_key"]].set_axis(["game_id", "variant", "elo", "diff", "pk", "at", "tk"], axis=1).assign(side="black")])
def rz(s):
    med = s.median(); mad = (s - med).abs().median()
    return 0.6745 * (s - med) / mad
long["elo_rz"] = long.groupby("variant").elo.transform(rz)
long["diff_rz"] = long.groupby("variant")["diff"].transform(rz)
print(long.groupby("variant").agg(n=("elo", "size"), elo_median=("elo", "median"), elo_mad=("elo", lambda s: (s - s.median()).abs().median()),
                                   diff_median=("diff", "median"), diff_mad=("diff", lambda s: (s - s.median()).abs().median())))
print("elo robust z < -3.5:", int((long.elo_rz < -3.5).sum()), "| > 3.5:", int((long.elo_rz > 3.5).sum()))
print("rating_diff |robust z| > 3.5:", int((long.diff_rz.abs() > 3.5).sum()), "of", len(long), " (mean ± 3 SD is NOT used)")
big = long[long.diff_rz.abs() > 3.5]
print("flagged diffs: range", big["diff"].min(), "to", big["diff"].max(), "| share with elo < 1000:", round((big.elo < 1000).mean(), 3), "vs overall", round((long.elo < 1000).mean(), 3))
print("most frequent elo values:", long.elo.value_counts().head(5).to_dict(), "(1500 = Lichess starting rating, inferred)")

             n  elo_median  elo_mad  diff_median  diff_mad
variant                                                   
Atomic     474      1550.5    222.5          0.0       4.0
Chess960   202      1854.0    196.0          0.0       4.0
Standard  8394      1863.0    204.0          0.0       4.0
elo robust z < -3.5: 21 | > 3.5: 0
rating_diff |robust z| > 3.5: 164 of 9070  (mean ± 3 SD is NOT used)
flagged diffs: range -424 to 393 | share with elo < 1000: 0.055 vs overall 0.008
most frequent elo values: {1500: 25, 1772: 24, 2012: 22, 1806: 21, 1785: 21} (1500 = Lichess starting rating, inferred)


In [10]:
# Are the big rating changes real? A player's next rating in the same tournament should equal previous rating + previous change.
pg = long.sort_values(["pk", "tk", "at"]).copy()
pg["prev_elo"] = pg.groupby(["pk", "tk"]).elo.shift(); pg["prev_diff"] = pg.groupby(["pk", "tk"])["diff"].shift()
c = pg.dropna(subset=["prev_elo"]); gap = c.elo - (c.prev_elo + c.prev_diff)
print("consecutive games compared:", len(c), "| next elo == prev elo + prev diff:", int((gap == 0).sum()), f"({(gap == 0).mean():.2%})", "| mismatches:", int((gap != 0).sum()))
print("mismatch size:", gap[gap != 0].describe().round(1).to_dict())
bigc = c[c.diff_rz.abs() > 3.5]
print("continuity inside the flagged big-diff rows' successors is checked via prev_diff above; flagged rows kept as unusual_large")

consecutive games compared: 7582 | next elo == prev elo + prev diff: 7565 (99.78%) | mismatches: 17
mismatch size: {'count': 17.0, 'mean': 1.5, 'std': 8.3, 'min': -12.0, '25%': -4.0, '50%': 3.0, '75%': 5.0, 'max': 22.0}
continuity inside the flagged big-diff rows' successors is checked via prev_diff above; flagged rows kept as unusual_large


## 5. Text and categories: repr() of distinct values, Unicode form, whitespace, case

In [11]:
def text_checks(df, cols):
    out = []
    for c in cols:
        s = df[c].dropna().astype(str)
        out.append({"column": c, "distinct": s.nunique(), "not_NFC": int((s != s.map(lambda x: unicodedata.normalize("NFC", x))).sum()),
                    "lead_trail_space": int((s != s.str.strip()).sum()), "nbsp": int(s.str.contains("\u00a0").sum()),
                    "double_space": int(s.str.contains("  ").sum()), "non_ascii": int(s.str.contains(r"[^\x00-\x7f]").sum()),
                    "case_variants": s.nunique() - s.str.casefold().nunique()})
    return pd.DataFrame(out)
display(text_checks(t, ["source_id", "full_name", "variant"]))
display(text_checks(g, ["game_id", "result", "termination"]))
display(text_checks(p, ["source_id"])); display(text_checks(o, ["eco", "name"]))
for df, c in [(t, "variant"), (g, "termination"), (g, "result")]:
    print(c, {repr(k): v for k, v in df[c].value_counts(dropna=False).items()})
print("tournament names with non-ASCII:", sorted(t[t.full_name.str.contains(r"[^\x00-\x7f]")].full_name.str.replace(r"\d+", "N", regex=True).unique()))
print("ECO codes not matching [A-E]dd:", int((~o.eco.str.fullmatch(r"[A-E]\d\d")).sum()))
print("player ids not matching [A-Za-z0-9_-]+:", int((~p.source_id.str.fullmatch(r"[A-Za-z0-9_-]+")).sum()))
print("moves: contains clock/comment markup:", int(g.moves.str.contains(r"\{|\[%").sum()), "| last token equals result:", int((g.moves.str.extract(r"(1-0|0-1|1/2-1/2|\*)\s*$")[0] == g.result).sum()), "of", len(g))
print("moves that are only the result token (no moves played):", int(g.moves.str.fullmatch(r"(1-0|0-1|1/2-1/2)").sum()), g[g.moves.str.fullmatch(r"(1-0|0-1|1/2-1/2)")].termination.value_counts().to_dict())

,column,distinct,not_NFC,lead_trail_space,nbsp,double_space,non_ascii,case_variants
0,source_id,278,0,0,0,0,0,0
1,full_name,50,0,0,0,0,87,0
2,variant,9,0,0,0,0,0,0


,column,distinct,not_NFC,lead_trail_space,nbsp,double_space,non_ascii,case_variants
0,game_id,4535,0,0,0,0,0,0
1,result,3,0,0,0,0,0,0
2,termination,3,0,0,0,0,0,0


,column,distinct,not_NFC,lead_trail_space,nbsp,double_space,non_ascii,case_variants
0,source_id,1307,0,0,0,0,0,0


,column,distinct,not_NFC,lead_trail_space,nbsp,double_space,non_ascii,case_variants
0,eco,500,0,0,0,0,0,0
1,name,3177,0,0,0,0,226,0


variant {"'Standard'": 176, "'Atomic'": 16, "'Three-check'": 15, "'Antichess'": 14, "'Crazyhouse'": 12, "'King of the Hill'": 12, "'Chess960'": 12, "'Racing Kings'": 11, "'Horde'": 10}
termination {"'Time forfeit'": 2474, "'Normal'": 1955, "'Abandoned'": 106}
result {"'1-0'": 2302, "'0-1'": 2128, "'1/2-1/2'": 105}
tournament names with non-ASCII: ['Grünfeld Defense Blitz Arena', 'Grünfeld Defense Bullet Arena', 'Grünfeld Defense Rapid Arena', 'Grünfeld Defense SuperBlitz Arena', '≤N Blitz Arena', '≤N Bullet Arena', '≤N Rapid Arena', '≤N SuperBlitz Arena']
ECO codes not matching [A-E]dd: 0
player ids not matching [A-Za-z0-9_-]+: 0
moves: contains clock/comment markup: 0 | last token equals result: 4535 of 4535
moves that are only the result token (no moves played): 55 {'Abandoned': 55}


## 2. Invalid values: time columns

In [12]:
now = pd.Timestamp.now("UTC").tz_localize(None)
for name, df, cols in [("tournament", t, ["starts_at", "finishes_at", "retrieved_at"]), ("game", g, ["played_at", "retrieved_at"])]:
    for c in cols:
        s = pd.to_datetime(df[c], errors="coerce", utc=True).dt.tz_convert(None)
        print(f"{name}.{c}: unparsable={int(s.isna().sum() - df[c].isna().sum())} min={s.min()} max={s.max()} future={int((s > now).sum())}")

tournament.starts_at: unparsable=0 min=2026-10-08 10:30:30 max=2026-10-10 00:00:40 future=0
tournament.finishes_at: unparsable=0 min=2026-10-08 10:57:30 max=2026-10-10 02:00:00 future=0
tournament.retrieved_at: unparsable=0 min=2026-10-08 12:06:52 max=2026-10-09 18:19:39 future=0
game.played_at: unparsable=0 min=2026-10-08 11:30:12 max=2026-10-09 17:55:47 future=0
game.retrieved_at: unparsable=0 min=2026-10-08 12:06:52 max=2026-10-09 18:20:30 future=0


## 4. Failures across rows and fields
Cross-field and cross-table consistency (none of these needs a model, only rules).

In [13]:
gt = g.merge(t[["tournament_key", "starts_at", "finishes_at", "minutes", "nb_players", "status", "max_rating"]], on="tournament_key")
dur = (pd.to_datetime(t.finishes_at) - pd.to_datetime(t.starts_at)).dt.total_seconds() / 60
print("finishes_at < starts_at:", int((t.finishes_at < t.starts_at).sum()))
print("duration != minutes (> 1 min):", int((abs(dur - t.minutes) > 1).sum()))
print("game played outside tournament window:", int(((gt.played_at < gt.starts_at) | (gt.played_at > gt.finishes_at)).sum()))
print("same player both colours:", int((g.white_player_key == g.black_player_key).sum()))
print("rating change sign contradicts result: winner < 0:", int(((g.result == "1-0") & (g.white_rating_diff < 0)).sum() + ((g.result == "0-1") & (g.black_rating_diff < 0)).sum()),
      "| loser > 0:", int(((g.result == "1-0") & (g.black_rating_diff > 0)).sum() + ((g.result == "0-1") & (g.white_rating_diff > 0)).sum()))
pp = pd.concat([g[["tournament_key", "white_player_key"]].set_axis(["tk", "pk"], axis=1), g[["tournament_key", "black_player_key"]].set_axis(["tk", "pk"], axis=1)])
chk = pp.groupby("tk").pk.nunique().rename("players_in_games").to_frame().join(t.set_index("tournament_key")[["nb_players", "status"]])
chk["registered_without_game"] = chk.nb_players - chk.players_in_games
print("nb_players < players seen in games:", int((chk.registered_without_game < 0).sum())); display(chk)
print("games above the tournament's rating cap (elo > max_rating):", int(((gt.max_rating.notna()) & ((gt.white_elo > gt.max_rating) | (gt.black_elo > gt.max_rating))).sum()), "(max overshoot:", int(max((gt.white_elo - gt.max_rating).max(), (gt.black_elo - gt.max_rating).max())), ")")
print(pd.crosstab(g.termination, g.result))
g["n_moves"] = g.moves.str.count(r"\d+\.")
print(g.groupby("termination").n_moves.describe()[["min", "50%", "max"]])
print("same player twice at the same second:", int(pg.duplicated(["pk", "at"]).sum()))

finishes_at < starts_at: 0
duration != minutes (> 1 min): 0
game played outside tournament window: 0
same player both colours: 0
rating change sign contradicts result: winner < 0: 0 | loser > 0: 0
nb_players < players seen in games: 0


,players_in_games,nb_players,status,registered_without_game
tk,,,,
117,105,113,30,8
118,38,40,30,2
119,343,364,30,21
120,60,64,30,4
121,81,82,30,1
260,455,463,30,8
273,406,423,30,17


games above the tournament's rating cap (elo > max_rating): 1 (max overshoot: 1 )
result         0-1   1-0  1/2-1/2
termination                      
Abandoned       55    51        0
Normal         888  1007       60
Time forfeit  1185  1244       45
              min   50%   max
termination                  
Abandoned     0.0   0.0   1.0
Normal        1.0  27.0  96.0
Time forfeit  1.0  34.0  85.0
same player twice at the same second: 0


In [14]:
# name vs clock: Lichess speed category by estimated duration (limit + 40 * increment)
def speed(r):
    est = r.clock_limit + 40 * r.clock_increment
    return "UltraBullet" if est < 30 else "Bullet" if est < 180 else "Blitz" if est < 480 else "Rapid" if est < 1500 else "Classical"
t["est_speed"] = t.apply(speed, axis=1)
def name_speed(n):
    for k in ("UltraBullet", "SuperBlitz", "Bullet", "Blitz", "Rapid", "Classical"):
        if k in n: return k
t["name_speed"] = t.full_name.map(name_speed)
x = t[t.name_speed.notna() & (t.name_speed != "SuperBlitz")]
print("name says one speed, clock says another:", int((x.name_speed != x.est_speed).sum()), "of", len(x), "(SuperBlitz excluded: not a Lichess category, all 180+0)")
print("clock_limit = 0 tournaments:", t[t.clock_limit == 0][["full_name", "clock_limit", "clock_increment"]].drop_duplicates().to_dict("records"))

name says one speed, clock says another: 0 of 146 (SuperBlitz excluded: not a Lichess category, all 180+0)
clock_limit = 0 tournaments: [{'full_name': 'Hourly Antichess Arena', 'clock_limit': 0, 'clock_increment': 2}, {'full_name': 'Hourly Atomic Arena', 'clock_limit': 0, 'clock_increment': 2}]


## 6. Identity: one thing as two, two as one

In [15]:
cf = p.source_id.map(lambda s: unicodedata.normalize("NFC", s).casefold())
print("player ids equal after NFC+casefold but different raw:", int(p.source_id.nunique() - cf.nunique()))
print("tournament duplicates by (full_name, starts_at):", int(t.duplicated(["full_name", "starts_at"]).sum()))
print("players that never appear in a game:", int((~p.player_key.isin(pd.concat([g.white_player_key, g.black_player_key]))).sum()))
dupn = o[o.name.isin(o.name[o.name.duplicated()])]
print("opening names listed under more than one ECO code:", dupn.name.nunique(), "| casefold-only variants:", int(o.name.nunique() - o.name.str.casefold().nunique()))
display(dupn.sort_values("name").head(6)[["opening_key", "eco", "name"]])
print("same PGN line under more than one opening:", int((ol.groupby("pgn").opening_key.nunique() > 1).sum()))
print("openings never used by a game:", int((~o.opening_key.isin(g.opening_key.dropna())).sum()), "of", len(o))

player ids equal after NFC+casefold but different raw: 0
tournament duplicates by (full_name, starts_at): 0
players that never appear in a game: 0
opening names listed under more than one ECO code: 108 | casefold-only variants: 0


,opening_key,eco,name
858,859,B03,Alekhine Defense
831,832,B02,Alekhine Defense
588,589,A57,Benko Gambit
608,609,A59,Benko Gambit
581,582,A56,Benoni Defense
613,614,A61,Benoni Defense


same PGN line under more than one opening: 0
openings never used by a game: 2721 of 3332


## 7. Checks that found nothing (exact queries, for the report)

In [16]:
checks = [
 ("2 invalid", "negative or zero elo",                 "SELECT COUNT(*) FROM game WHERE white_elo <= 0 OR black_elo <= 0"),
 ("2 invalid", "sentinel elo (-1, 9999, 99999)",       "SELECT COUNT(*) FROM game WHERE white_elo IN (-1,9999,99999) OR black_elo IN (-1,9999,99999)"),
 ("2 invalid", "negative nb_players / minutes / clock", "SELECT COUNT(*) FROM tournament WHERE nb_players < 0 OR minutes <= 0 OR clock_limit < 0 OR clock_increment < 0"),
 ("2 invalid", "clock 0+0",                            "SELECT COUNT(*) FROM tournament WHERE clock_limit = 0 AND clock_increment = 0"),
 ("2 invalid", "finishes_at before starts_at",         "SELECT COUNT(*) FROM tournament WHERE finishes_at < starts_at"),
 ("2 invalid", "game played outside its tournament",   "SELECT COUNT(*) FROM game g JOIN tournament t USING (tournament_key) WHERE g.played_at < t.starts_at OR g.played_at > t.finishes_at"),
 ("2 invalid", "game timestamp in the future",         "SELECT COUNT(*) FROM game WHERE played_at > strftime('%Y-%m-%dT%H:%M:%SZ','now')"),
 ("2 invalid", "ECO code not A00-E99",                 "SELECT COUNT(*) FROM opening WHERE eco NOT GLOB '[A-E][0-9][0-9]'"),
 ("1 missing", "empty strings in text columns",        "SELECT COUNT(*) FROM game WHERE TRIM(termination)='' OR TRIM(moves)=''"),
 ("1 missing", "NULL elo / rating_diff / termination / moves", "SELECT COUNT(*) FROM game WHERE white_elo IS NULL OR black_elo IS NULL OR white_rating_diff IS NULL OR black_rating_diff IS NULL OR termination IS NULL OR moves IS NULL"),
 ("4 sequence", "same player on both sides of a game", "SELECT COUNT(*) FROM game WHERE white_player_key = black_player_key"),
 ("4 sequence", "winner loses rating / loser gains",   "SELECT COUNT(*) FROM game WHERE (result='1-0' AND (white_rating_diff<0 OR black_rating_diff>0)) OR (result='0-1' AND (black_rating_diff<0 OR white_rating_diff>0))"),
 ("4 sequence", "nb_players below players seen in games", "SELECT COUNT(*) FROM tournament t JOIN (SELECT tournament_key, COUNT(DISTINCT pk) n FROM (SELECT tournament_key, white_player_key pk FROM game UNION SELECT tournament_key, black_player_key FROM game) GROUP BY tournament_key) x USING (tournament_key) WHERE t.nb_players < x.n"),
 ("5 text", "variant/termination case or spelling variants", "SELECT COUNT(*) FROM (SELECT LOWER(variant) v FROM tournament GROUP BY LOWER(variant) HAVING COUNT(DISTINCT variant) > 1)"),
 ("6 identity", "players equal ignoring case",         "SELECT COUNT(*) FROM (SELECT LOWER(source_id) FROM player GROUP BY LOWER(source_id) HAVING COUNT(*) > 1)"),
 ("6 identity", "duplicate tournaments (name+start)",  "SELECT COUNT(*) FROM (SELECT 1 FROM tournament GROUP BY full_name, starts_at HAVING COUNT(*) > 1)"),
 ("3/4", "foreign_key_check rows",                     "SELECT COUNT(*) FROM pragma_foreign_key_check"),
]
res = pd.DataFrame([{"family": f, "check": c, "result": con.execute(sql).fetchone()[0], "sql": sql} for f, c, sql in checks])
res[["family", "check", "result"]]

,family,check,result
0,2 invalid,negative or zero elo,0
1,2 invalid,"sentinel elo (-1, 9999, 99999)",0
2,2 invalid,negative nb_players / minutes / clock,0
3,2 invalid,clock 0+0,0
4,2 invalid,finishes_at before starts_at,0
5,2 invalid,game played outside its tournament,0
6,2 invalid,game timestamp in the future,0
7,2 invalid,ECO code not A00-E99,0
8,1 missing,empty strings in text columns,0
9,1 missing,NULL elo / rating_diff / termination / moves,0
